# LLM-as-Judge — Métrica de Satisfacción de Justificaciones
## PROJENER.AI SL · UAX · Máster Big Data · 2026

**Autora:** Edurne Martínez de Contrasta  
**Propósito:** Evaluar la calidad de las justificaciones generadas por cada variante del sistema  
mediante un juez LLM externo (Zheng et al., 2023 — MT-Bench / LLM-as-Judge).  
**Referencia metodológica:** Zheng et al. (2023). *Judging LLM-as-a-Judge with MT-Bench.*  

---

## Dimensiones de evaluación (escala 1–5)

| Dimensión | Definición operativa |
|-----------|---------------------|
| **Claridad** | ¿La justificación es fácil de entender para un responsable de procurement sin conocimientos técnicos de IA? |
| **Completitud** | ¿Menciona todos los factores relevantes del caso (proveedor, presupuesto, normativa, riesgo contractual)? |
| **Accionabilidad** | ¿Permite al receptor tomar una decisión clara o saber qué hacer a continuación? |

**Puntuación de Satisfacción (SAT):** media aritmética de las tres dimensiones (rango 1.0–5.0).  
Objetivo mínimo: SAT > 3.5 para considerar la justificación útil en un entorno empresarial real.

---

## Variantes evaluadas

Se evalúan las 4 variantes principales que generan justificaciones textuales:  
- **RPA-Baseline**: justificaciones deterministas basadas en reglas  
- **Single-Small**: agente único llama-3.1-8b  
- **Multi-Small**: 5 agentes especializados llama-3.1-8b  
- **Multi-Mixed**: configuración óptima (8b×4 + 70b×1)

## 1. Justificaciones representativas por variante

Se seleccionan 15 casos representativos del experimento principal (5 Simple, 5 Medio, 5 Complejo)  
con las justificaciones reales generadas por cada variante.  
Esto cubre los tres niveles de complejidad y es suficiente para obtener una estimación  
estadísticamente robusta del SAT por variante (n=15 × 4 variantes = 60 evaluaciones).

In [1]:
# Justificaciones reales extraídas del experimento principal
# Campos: caso_id, nivel, variante, decision, justificacion, requiere_hil (ground truth)

justificaciones = [
    # === CASOS SIMPLES (C01–C05) ===
    # C01: Compra estándar material oficina 150€, proveedor OK
    {"caso_id": "C01", "nivel": "Simple", "variante": "RPA-Baseline",
     "decision": "aprobado", "requiere_hil": False,
     "justificacion": "Proveedor OK + presupuesto OK. Categoría no regulada."},
    {"caso_id": "C01", "nivel": "Simple", "variante": "Single-Small",
     "decision": "aprobado", "requiere_hil": False,
     "justificacion": "Compra de material de oficina por 150€. El proveedor OfficeMax está aprobado y activo. El presupuesto del departamento administrativo tiene fondos disponibles. La categoría material_oficina no requiere normativa especial. Aprobación directa recomendada."},
    {"caso_id": "C01", "nivel": "Simple", "variante": "Multi-Small",
     "decision": "aprobado", "requiere_hil": False,
     "justificacion": "Requester: solicitud válida. Procurement: proveedor homologado sin alertas. Finance: presupuesto disponible, importe dentro del límite automático. Legal: sin cláusulas de riesgo. Compliance: categoría no regulada. Decisión: aprobado."},
    {"caso_id": "C01", "nivel": "Simple", "variante": "Multi-Mixed",
     "decision": "aprobado", "requiere_hil": False,
     "justificacion": "Solicitud de material de oficina (150€) procesada satisfactoriamente. Verificaciones completadas: proveedor activo y homologado, presupuesto disponible en el departamento, categoría sin requerimientos regulatorios especiales, contrato estándar sin cláusulas de riesgo. Aprobación automática procedente."},

    # C05: Compra equipamiento IT 800€, proveedor OK
    {"caso_id": "C05", "nivel": "Simple", "variante": "RPA-Baseline",
     "decision": "aprobado", "requiere_hil": False,
     "justificacion": "Proveedor OK + presupuesto OK. Categoría no regulada."},
    {"caso_id": "C05", "nivel": "Simple", "variante": "Single-Small",
     "decision": "aprobado", "requiere_hil": False,
     "justificacion": "Equipamiento IT por 800€. Proveedor TechSupply aprobado. Presupuesto IT disponible. Sin restricciones normativas para esta categoría. Aprobado."},
    {"caso_id": "C05", "nivel": "Simple", "variante": "Multi-Small",
     "decision": "aprobado", "requiere_hil": False,
     "justificacion": "Finance: fondos disponibles. Legal: contrato estándar. Compliance: sin alertas regulatorias. Aprobado."},
    {"caso_id": "C05", "nivel": "Simple", "variante": "Multi-Mixed",
     "decision": "aprobado", "requiere_hil": False,
     "justificacion": "Compra de equipamiento IT (800€) aprobada. Proveedor verificado, presupuesto confirmado, sin incidencias normativas ni contractuales."},

    # C10: Material laboratorio 2.500€, proveedor OK
    {"caso_id": "C10", "nivel": "Simple", "variante": "RPA-Baseline",
     "decision": "aprobado", "requiere_hil": False,
     "justificacion": "Proveedor OK + presupuesto OK. Categoría no regulada."},
    {"caso_id": "C10", "nivel": "Simple", "variante": "Single-Small",
     "decision": "aprobado", "requiere_hil": False,
     "justificacion": "Material de laboratorio 2.500€. Proveedor LabTech verificado. Presupuesto I+D disponible. Sin normativa específica aplicable. Aprobado automáticamente."},
    {"caso_id": "C10", "nivel": "Simple", "variante": "Multi-Small",
     "decision": "aprobado", "requiere_hil": False,
     "justificacion": "Procurement: proveedor OK. Finance: presupuesto OK. Legal: sin cláusulas especiales. Compliance: categoría no regulada. Aprobado."},
    {"caso_id": "C10", "nivel": "Simple", "variante": "Multi-Mixed",
     "decision": "aprobado", "requiere_hil": False,
     "justificacion": "Adquisición de material de laboratorio (2.500€) aprobada. Todas las verificaciones superadas sin incidencias."},

    # C15: Servicio limpieza 1.200€/mes, proveedor nuevo
    {"caso_id": "C15", "nivel": "Simple", "variante": "RPA-Baseline",
     "decision": "rechazado", "requiere_hil": False,
     "justificacion": "Proveedor LimpiezaTotal no aprobado. RPA no puede gestionar altas."},
    {"caso_id": "C15", "nivel": "Simple", "variante": "Single-Small",
     "decision": "rechazado", "requiere_hil": False,
     "justificacion": "El proveedor LimpiezaTotal no figura en el registro de proveedores homologados. Rechazado hasta completar proceso de alta de proveedor."},
    {"caso_id": "C15", "nivel": "Simple", "variante": "Multi-Small",
     "decision": "rechazado", "requiere_hil": False,
     "justificacion": "Procurement: proveedor no homologado, estado=pendiente_alta. Rechazado. Acción requerida: iniciar proceso de homologación."},
    {"caso_id": "C15", "nivel": "Simple", "variante": "Multi-Mixed",
     "decision": "rechazado", "requiere_hil": False,
     "justificacion": "Solicitud rechazada: el proveedor LimpiezaTotal no está homologado (estado: pendiente_alta). Para proceder, iniciar el proceso de alta de proveedor en el sistema de procurement. Tiempo estimado: 5-10 días laborables."},

    # C18: Catering evento 3.000€, proveedor OK presupuesto justo
    {"caso_id": "C18", "nivel": "Simple", "variante": "RPA-Baseline",
     "decision": "rechazado", "requiere_hil": False,
     "justificacion": "Presupuesto insuficiente. Déficit: 200€. RPA no puede reasignar."},
    {"caso_id": "C18", "nivel": "Simple", "variante": "Single-Small",
     "decision": "rechazado", "requiere_hil": False,
     "justificacion": "Presupuesto de eventos insuficiente para cubrir los 3.000€ del catering. Déficit de 200€. Rechazado por falta de fondos."},
    {"caso_id": "C18", "nivel": "Simple", "variante": "Multi-Small",
     "decision": "rechazado", "requiere_hil": False,
     "justificacion": "Finance: presupuesto_disponible < importe_solicitado (déficit 200€). Rechazado por insuficiencia presupuestaria."},
    {"caso_id": "C18", "nivel": "Simple", "variante": "Multi-Mixed",
     "decision": "rechazado", "requiere_hil": False,
     "justificacion": "Catering para evento rechazado por insuficiencia presupuestaria (déficit: 200€). Opciones: (1) solicitar suplemento presupuestario, (2) reducir el importe del catering, (3) aplazar al siguiente ejercicio."},

    # === CASOS MEDIOS (C22–C26) ===
    # C22: Software SaaS RRHH con datos personales empleados (GDPR)
    {"caso_id": "C22", "nivel": "Medio", "variante": "RPA-Baseline",
     "decision": "aprobado", "requiere_hil": True,
     "justificacion": "Proveedor OK + presupuesto OK. Categoría no regulada."},
    {"caso_id": "C22", "nivel": "Medio", "variante": "Single-Small",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Software de RRHH que procesará datos personales de empleados. Detectada posible aplicación de GDPR. Escalo para revisión legal."},
    {"caso_id": "C22", "nivel": "Medio", "variante": "Multi-Small",
     "decision": "aprobado", "requiere_hil": True,
     "justificacion": "Proveedor OK. Finance: aprobado. Compliance: cumple normativa. Aprobado."},
    {"caso_id": "C22", "nivel": "Medio", "variante": "Multi-Mixed",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Software SaaS de RRHH requiere revisión legal prioritaria: (1) procesamiento de datos personales de empleados activa GDPR Art. 28 — necesario DPA con el proveedor; (2) transferencia a servidores fuera de la UE requiere cláusulas contractuales tipo. Escalo al responsable de privacidad y legal."},

    # C25: Consultoría estratégica 45.000€, proveedor con conflicto de interés
    {"caso_id": "C25", "nivel": "Medio", "variante": "RPA-Baseline",
     "decision": "aprobado", "requiere_hil": True,
     "justificacion": "Proveedor OK + presupuesto OK. Categoría no regulada."},
    {"caso_id": "C25", "nivel": "Medio", "variante": "Single-Small",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Conflicto de interés detectado: proveedor vinculado con directivo. Escalo."},
    {"caso_id": "C25", "nivel": "Medio", "variante": "Multi-Small",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Procurement: alerta conflicto_interes=True. Legal: riesgo contractual alto. Escalo a dirección."},
    {"caso_id": "C25", "nivel": "Medio", "variante": "Multi-Mixed",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Contrato de consultoría (45.000€) escalado por conflicto de interés: el proveedor ConsultingPro tiene vinculación directa con el Director de Operaciones (participación accionarial). Requiere aprobación del Comité de Ética y posible abstención del directivo afectado en la decisión."},

    # C28: Compra dron industrial, requiere licencia AESA
    {"caso_id": "C28", "nivel": "Medio", "variante": "RPA-Baseline",
     "decision": "aprobado", "requiere_hil": True,
     "justificacion": "Proveedor OK + presupuesto OK. Categoría no regulada."},
    {"caso_id": "C28", "nivel": "Medio", "variante": "Single-Small",
     "decision": "aprobado", "requiere_hil": True,
     "justificacion": "Dron industrial aprobado. Proveedor y presupuesto OK."},
    {"caso_id": "C28", "nivel": "Medio", "variante": "Multi-Small",
     "decision": "aprobado", "requiere_hil": True,
     "justificacion": "Compliance: categoría drones verificada. Sin alertas. Aprobado."},
    {"caso_id": "C28", "nivel": "Medio", "variante": "Multi-Mixed",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Adquisición de dron industrial escalada: la normativa AESA (RD 1088/2021) exige operador certificado y registro del aparato antes del uso. Verificar que la empresa dispone de certificación de operador de UAS antes de aprobar la compra."},

    # C31: Renovación contrato mantenimiento con cláusula penalización
    {"caso_id": "C31", "nivel": "Medio", "variante": "RPA-Baseline",
     "decision": "aprobado", "requiere_hil": True,
     "justificacion": "Proveedor OK + presupuesto OK + normativa OK."},
    {"caso_id": "C31", "nivel": "Medio", "variante": "Single-Small",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Cláusula de penalización elevada detectada. Escalo a legal."},
    {"caso_id": "C31", "nivel": "Medio", "variante": "Multi-Small",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Legal: penalizacion_maxima=85.000€ supera umbral. Riesgo contractual alto. Escalo."},
    {"caso_id": "C31", "nivel": "Medio", "variante": "Multi-Mixed",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Renovación de contrato de mantenimiento escalada: cláusula de penalización por incumplimiento de SLA cifrada en 85.000€ (≈70% del valor anual del contrato). Recomendación: negociar cap máximo de penalización al 30% del valor contractual antes de firmar."},

    # C35: Outsourcing RRHH parcial, importe 120.000€/año
    {"caso_id": "C35", "nivel": "Medio", "variante": "RPA-Baseline",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Normativa no cumplida: outsourcing_rrhh. Escalado sin contexto."},
    {"caso_id": "C35", "nivel": "Medio", "variante": "Single-Small",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Outsourcing de RRHH por encima del umbral de Consejo (>100k€). Escalo a dirección."},
    {"caso_id": "C35", "nivel": "Medio", "variante": "Multi-Small",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Finance: importe_consejo=True (>100k€). Compliance: outsourcing_rrhh requiere aprobación especial. Escalo a Consejo."},
    {"caso_id": "C35", "nivel": "Medio", "variante": "Multi-Mixed",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Outsourcing de RRHH (120.000€/año) requiere aprobación del Consejo de Administración por dos motivos independientes: (1) importe supera el umbral delegado (>100.000€); (2) decisión estratégica con impacto en plantilla. Preparar informe ejecutivo con análisis coste-beneficio para próxima reunión del Consejo."},

    # === CASOS COMPLEJOS (C40–C44) ===
    # C40: Contrato consignación, importe 0€, sin contrato marco
    {"caso_id": "C40", "nivel": "Complejo", "variante": "RPA-Baseline",
     "decision": "aprobado", "requiere_hil": True,
     "justificacion": "Proveedor OK + presupuesto OK. Categoría no regulada."},
    {"caso_id": "C40", "nivel": "Complejo", "variante": "Single-Small",
     "decision": "rechazado", "requiere_hil": True,
     "justificacion": "Importe 0€ — no hay gasto, se puede aprobar directamente."},
    {"caso_id": "C40", "nivel": "Complejo", "variante": "Multi-Small",
     "decision": "aprobado", "requiere_hil": True,
     "justificacion": "Finance: importe=0, aprobado. Compliance: sin alertas. Aprobado."},
    {"caso_id": "C40", "nivel": "Complejo", "variante": "Multi-Mixed",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Contrato de consignación escalado: aunque el importe inicial es 0€, este modelo contractual genera obligaciones de pago diferidas sin contrato marco previo que las regule. Riesgo legal: la empresa queda expuesta a reclamaciones sin límite de importe acordado. Requiere negociación de contrato marco antes de activar la consignación."},

    # C43: Adquisición empresa filial extranjera, importe 2.3M€
    {"caso_id": "C43", "nivel": "Complejo", "variante": "RPA-Baseline",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Normativa no cumplida. Escalado sin contexto."},
    {"caso_id": "C43", "nivel": "Complejo", "variante": "Single-Small",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Importe 2.3M€ supera umbral de Consejo. Escalo."},
    {"caso_id": "C43", "nivel": "Complejo", "variante": "Multi-Small",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Finance: importe_consejo=True. Legal: adquisicion_empresa requiere due diligence. Escalo."},
    {"caso_id": "C43", "nivel": "Complejo", "variante": "Multi-Mixed",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Adquisición de filial extranjera (2.3M€) escalada como decisión estratégica crítica: requiere (1) aprobación unánime del Consejo de Administración, (2) due diligence legal y financiera completa, (3) posible notificación a autoridades de competencia si supera umbrales regulatorios, (4) valoración independiente por auditor externo."},

    # C46: Producto químico sin ficha técnica completa
    {"caso_id": "C46", "nivel": "Complejo", "variante": "RPA-Baseline",
     "decision": "aprobado", "requiere_hil": True,
     "justificacion": "Proveedor OK + presupuesto OK + normativa OK."},
    {"caso_id": "C46", "nivel": "Complejo", "variante": "Single-Small",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Producto químico sin documentación REACH completa. Escalo a seguridad."},
    {"caso_id": "C46", "nivel": "Complejo", "variante": "Multi-Small",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Compliance: reach_violation=True. Riesgo regulatorio alto. Escalo."},
    {"caso_id": "C46", "nivel": "Complejo", "variante": "Multi-Mixed",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Compra de producto químico bloqueada: incumplimiento del Reglamento REACH (CE 1907/2006). La ficha de datos de seguridad está incompleta — faltan secciones 8 (EPIs requeridos) y 14 (transporte). No puede aprobarse hasta recibir ficha técnica completa y validación del responsable de PRL."},

    # C48: Contrato exclusividad proveedor único, 5 años
    {"caso_id": "C48", "nivel": "Complejo", "variante": "RPA-Baseline",
     "decision": "aprobado", "requiere_hil": True,
     "justificacion": "Proveedor OK + presupuesto OK + normativa OK."},
    {"caso_id": "C48", "nivel": "Complejo", "variante": "Single-Small",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Contrato de exclusividad 5 años, riesgo legal alto. Escalo."},
    {"caso_id": "C48", "nivel": "Complejo", "variante": "Multi-Small",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Legal: clausula_exclusividad=True, duración 5 años. Riesgo dependencia proveedor. Escalo a dirección."},
    {"caso_id": "C48", "nivel": "Complejo", "variante": "Multi-Mixed",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Contrato de exclusividad a 5 años escalado: genera dependencia crítica de proveedor único con riesgo estratégico a largo plazo. Análisis requerido: (1) cláusula de salida y condiciones de resolución anticipada, (2) riesgo de concentración si el proveedor deteriora calidad o aumenta precios, (3) posible infracción de normativa de competencia si el proveedor tiene posición dominante en el mercado."},

    # C50: Proyecto CPD propio vs. nube, 500.000€ inversión
    {"caso_id": "C50", "nivel": "Complejo", "variante": "RPA-Baseline",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Normativa no cumplida. Escalado sin contexto."},
    {"caso_id": "C50", "nivel": "Complejo", "variante": "Single-Small",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Inversión 500k€, decisión estratégica. Escalo a Consejo."},
    {"caso_id": "C50", "nivel": "Complejo", "variante": "Multi-Small",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Finance: importe_consejo=True (500k€). Legal: contrato infraestructura crítica. Escalo."},
    {"caso_id": "C50", "nivel": "Complejo", "variante": "Multi-Mixed",
     "decision": "escalado_hil", "requiere_hil": True,
     "justificacion": "Inversión en CPD propio (500.000€) escalada como decisión estratégica de infraestructura: (1) importe requiere aprobación del Consejo; (2) decisión build vs. buy tiene implicaciones a 10 años en CAPEX/OPEX; (3) requiere análisis TCO comparativo con alternativas cloud; (4) impacto en política de datos y soberanía tecnológica. Recomendar comité técnico-financiero antes de decisión."},
]

print(f"Total justificaciones cargadas: {len(justificaciones)}")
variantes = list(dict.fromkeys(j['variante'] for j in justificaciones))
niveles = list(dict.fromkeys(j['nivel'] for j in justificaciones))
print(f"Variantes: {variantes}")
print(f"Niveles: {niveles}")
print(f"Casos únicos: {len(set(j['caso_id'] for j in justificaciones))}")

Total justificaciones cargadas: 60
Variantes: ['RPA-Baseline', 'Single-Small', 'Multi-Small', 'Multi-Mixed']
Niveles: ['Simple', 'Medio', 'Complejo']
Casos únicos: 15


## 2. Evaluación LLM-as-Judge via Ollama (Mistral local)

El juez evalúa cada justificación en las tres dimensiones (Claridad, Completitud, Accionabilidad)  
en escala 1–5 y devuelve JSON estructurado.  

Se usa `mistral` ejecutado localmente via Ollama como juez externo independiente  
al sistema evaluado (que usó modelos Llama vía Groq), evitando sesgos de auto-evaluación.  
Esta elección sigue la metodología de Zheng et al. (2023): el juez debe ser un modelo  
distinto a los sistemas evaluados.

In [2]:
import json
import time
import requests

# --- Ollama corre en local, sin API key ---
OLLAMA_URL = "http://localhost:11434/api/chat"
JUDGE_MODEL = "mistral"

JUDGE_PROMPT_SYSTEM = """Eres un evaluador experto en procesos de procurement empresarial.
Tu tarea es evaluar la calidad de justificaciones generadas automáticamente por sistemas de IA
para decisiones de procurement.

Evalúa ÚNICAMENTE la justificación en tres dimensiones, en escala 1-5:
- claridad (1=incomprensible, 5=perfectamente claro para un responsable no técnico)
- completitud (1=omite factores clave, 5=menciona todos los factores relevantes del caso)
- accionabilidad (1=no permite tomar decisión, 5=indica claramente qué hacer a continuación)

Responde SOLO con JSON válido, sin texto adicional, sin markdown:
{\"claridad\": X, \"completitud\": X, \"accionabilidad\": X, \"comentario\": \"frase breve\"}"""


def evaluar_justificacion(caso_id, nivel, variante, decision, justificacion, requiere_hil):
    """Llama a Mistral local via Ollama como juez."""
    contexto = f"""Caso: {caso_id} | Nivel: {nivel} | Sistema: {variante}
Decisión tomada: {decision} | Requería HiL según ground truth: {requiere_hil}

Justificación a evaluar:
{justificacion}"""

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": JUDGE_MODEL,
            "stream": False,
            "options": {"temperature": 0.0},
            "messages": [
                {"role": "system", "content": JUDGE_PROMPT_SYSTEM},
                {"role": "user", "content": contexto}
            ]
        },
        timeout=120
    )

    if response.status_code != 200:
        raise Exception(f"Ollama error {response.status_code}: {response.text}")

    data = response.json()
    raw = data["message"]["content"].strip()

    # Limpiar posibles backticks
    raw = raw.replace("```json", "").replace("```", "").strip()
    resultado = json.loads(raw)

    sat = round((resultado["claridad"] + resultado["completitud"] + resultado["accionabilidad"]) / 3, 3)
    resultado["SAT"] = sat
    resultado["caso_id"] = caso_id
    resultado["nivel"] = nivel
    resultado["variante"] = variante
    resultado["decision"] = decision
    resultado["requiere_hil"] = requiere_hil
    resultado["justificacion"] = justificacion

    return resultado


# Verificar que Ollama está corriendo
try:
    r = requests.get("http://localhost:11434", timeout=5)
    print("✅ Ollama está corriendo correctamente")
except Exception as e:
    print(f"❌ Ollama no responde: {e}")
    print("   Asegúrate de que la app de Ollama está abierta en Windows")

print(f"Juez: {JUDGE_MODEL} (Ollama local) — independiente de los modelos evaluados (Llama/Groq)")
print(f"Evaluaciones a realizar: {len(justificaciones)}")

✅ Ollama está corriendo correctamente
Juez: mistral (Ollama local) — independiente de los modelos evaluados (Llama/Groq)
Evaluaciones a realizar: 60


In [3]:
import os

# Ejecutar evaluaciones
resultados_judge = []
errores = []

print("Iniciando evaluación LLM-as-Judge...")
print("=" * 60)

for i, j in enumerate(justificaciones):
    try:
        res = evaluar_justificacion(
            j["caso_id"], j["nivel"], j["variante"],
            j["decision"], j["justificacion"], j["requiere_hil"]
        )
        resultados_judge.append(res)
        print(f"[{i+1:2d}/{len(justificaciones)}] {j['caso_id']} | {j['variante']:20s} | "
              f"C={res['claridad']} Co={res['completitud']} A={res['accionabilidad']} → SAT={res['SAT']:.2f}")
        time.sleep(1.5)  # Rate limit
    except Exception as e:
        errores.append({"caso": j["caso_id"], "variante": j["variante"], "error": str(e)})
        print(f"  ❌ Error en {j['caso_id']}/{j['variante']}: {e}")

print("=" * 60)
print(f"Evaluaciones completadas: {len(resultados_judge)}/{len(justificaciones)}")
if errores:
    print(f"Errores: {len(errores)}")

Iniciando evaluación LLM-as-Judge...
[ 1/60] C01 | RPA-Baseline         | C=4 Co=3 A=2 → SAT=3.00
[ 2/60] C01 | Single-Small         | C=4 Co=5 A=4 → SAT=4.33
[ 3/60] C01 | Multi-Small          | C=4 Co=5 A=5 → SAT=4.67
[ 4/60] C01 | Multi-Mixed          | C=4 Co=5 A=5 → SAT=4.67
[ 5/60] C05 | RPA-Baseline         | C=4 Co=3 A=2 → SAT=3.00
[ 6/60] C05 | Single-Small         | C=4 Co=3 A=4 → SAT=3.67
[ 7/60] C05 | Multi-Small          | C=4 Co=3 A=4 → SAT=3.67
[ 8/60] C05 | Multi-Mixed          | C=5 Co=4 A=4 → SAT=4.33
[ 9/60] C10 | RPA-Baseline         | C=4 Co=3 A=2 → SAT=3.00
[10/60] C10 | Single-Small         | C=4 Co=3 A=5 → SAT=4.00
[11/60] C10 | Multi-Small          | C=4 Co=3 A=4 → SAT=3.67
[12/60] C10 | Multi-Mixed          | C=4 Co=3 A=4 → SAT=3.67
[13/60] C15 | RPA-Baseline         | C=4 Co=3 A=2 → SAT=3.00
[14/60] C15 | Single-Small         | C=5 Co=4 A=5 → SAT=4.67
  ❌ Error en C15/Multi-Small: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout

## 3. Resultados por Variante — Puntuación SAT media

In [4]:
from collections import defaultdict
import statistics

# Agrupar por variante
por_variante = defaultdict(list)
for r in resultados_judge:
    por_variante[r["variante"]].append(r)

# Calcular estadísticas
print("=" * 70)
print("  RESULTADOS LLM-as-Judge — Puntuación de Satisfacción (SAT)")
print("=" * 70)
print(f"{'Variante':25s} {'Claridad':9s} {'Complet.':9s} {'Accion.':9s} {'SAT':8s} {'n':4s}")
print("-" * 70)

resumen_variantes = {}
orden = ["RPA-Baseline", "Single-Small", "Multi-Small", "Multi-Mixed"]

for variante in orden:
    if variante not in por_variante:
        continue
    casos = por_variante[variante]
    c  = round(statistics.mean(r["claridad"] for r in casos), 2)
    co = round(statistics.mean(r["completitud"] for r in casos), 2)
    a  = round(statistics.mean(r["accionabilidad"] for r in casos), 2)
    sat = round(statistics.mean(r["SAT"] for r in casos), 2)
    sd  = round(statistics.stdev(r["SAT"] for r in casos), 2) if len(casos) > 1 else 0
    resumen_variantes[variante] = {"claridad": c, "completitud": co, "accionabilidad": a,
                                    "SAT": sat, "SD": sd, "n": len(casos)}
    ok = "✅" if sat >= 3.5 else "⚠️ "
    print(f"  {ok} {variante:22s} {c:9.2f} {co:9.2f} {a:9.2f} {sat:6.2f}±{sd:.2f}  {len(casos):3d}")

print("=" * 70)
print("  Objetivo mínimo SAT > 3.5 para uso empresarial")
print("=" * 70)

  RESULTADOS LLM-as-Judge — Puntuación de Satisfacción (SAT)
Variante                  Claridad  Complet.  Accion.   SAT      n   
----------------------------------------------------------------------
  ⚠️  RPA-Baseline                3.67      3.20      2.40   3.09±0.67   15
  ✅ Single-Small                4.20      3.60      4.00   3.93±0.69   15
  ✅ Multi-Small                 4.07      3.86      4.14   4.02±0.55   14
  ✅ Multi-Mixed                 4.27      4.53      4.27   4.36±0.27   15
  Objetivo mínimo SAT > 3.5 para uso empresarial


## 4. Análisis por Nivel de Complejidad

In [5]:
# SAT por variante y nivel de complejidad
print("SAT medio por variante y nivel de complejidad:")
print(f"{'Variante':25s} {'Simple':8s} {'Medio':8s} {'Complejo':9s}")
print("-" * 55)

for variante in orden:
    if variante not in por_variante:
        continue
    casos = por_variante[variante]
    por_nivel = defaultdict(list)
    for r in casos:
        por_nivel[r["nivel"]].append(r["SAT"])
    
    s = round(statistics.mean(por_nivel.get("Simple", [0])), 2)
    m = round(statistics.mean(por_nivel.get("Medio", [0])), 2)
    c = round(statistics.mean(por_nivel.get("Complejo", [0])), 2)
    print(f"  {variante:23s} {s:8.2f} {m:8.2f} {c:9.2f}")

SAT medio por variante y nivel de complejidad:
Variante                  Simple   Medio    Complejo 
-------------------------------------------------------


## 5. Guardar resultados y generar texto para el paper

In [6]:
import os

os.makedirs("resultados", exist_ok=True)

output = {
    "metadatos": {
        "juez": "mistral (Ollama local)",
        "metodologia": "LLM-as-Judge (Zheng et al., 2023)",
        "n_evaluaciones": len(resultados_judge),
        "n_casos": 15,
        "n_variantes": 4,
        "dimensiones": ["claridad", "completitud", "accionabilidad"],
        "escala": "1-5",
        "objetivo_SAT": 3.5
    },
    "resumen_por_variante": resumen_variantes,
    "evaluaciones_detalladas": resultados_judge
}

with open("resultados/llm_judge_satisfaccion.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print("✅ Resultados guardados en resultados/llm_judge_satisfaccion.json")
print()
print("=" * 60)
print("  TEXTO PARA EL PAPER (Sección Métricas)")
print("=" * 60)

mejor = max(resumen_variantes.items(), key=lambda x: x[1]["SAT"])
peor  = min(resumen_variantes.items(), key=lambda x: x[1]["SAT"])

print(f"""
Métrica SAT (Satisfaction Score):
Se evaluaron las justificaciones generadas por cada variante mediante
LLM-as-Judge (Zheng et al., 2023), usando mistral (Ollama local, Mistral AI) como evaluador
externo independiente sobre 15 casos representativos (5 Simple, 5 Medio,
5 Complejo). El juez evaluó tres dimensiones (claridad, completitud,
accionabilidad) en escala 1-5; SAT es su media aritmética.

Resultados:
- RPA-Baseline:  SAT = {resumen_variantes.get('RPA-Baseline', {}).get('SAT', 'N/A')} ± {resumen_variantes.get('RPA-Baseline', {}).get('SD', 'N/A')}
- Single-Small:  SAT = {resumen_variantes.get('Single-Small', {}).get('SAT', 'N/A')} ± {resumen_variantes.get('Single-Small', {}).get('SD', 'N/A')}
- Multi-Small:   SAT = {resumen_variantes.get('Multi-Small', {}).get('SAT', 'N/A')} ± {resumen_variantes.get('Multi-Small', {}).get('SD', 'N/A')}
- Multi-Mixed:   SAT = {resumen_variantes.get('Multi-Mixed', {}).get('SAT', 'N/A')} ± {resumen_variantes.get('Multi-Mixed', {}).get('SD', 'N/A')}

Variante con mayor SAT: {mejor[0]} (SAT={mejor[1]['SAT']})
Variante con menor SAT: {peor[0]} (SAT={peor[1]['SAT']})
""")

✅ Resultados guardados en resultados/llm_judge_satisfaccion.json

  TEXTO PARA EL PAPER (Sección Métricas)

Métrica SAT (Satisfaction Score):
Se evaluaron las justificaciones generadas por cada variante mediante
LLM-as-Judge (Zheng et al., 2023), usando mistral (Ollama local, Mistral AI) como evaluador
externo independiente sobre 15 casos representativos (5 Simple, 5 Medio,
5 Complejo). El juez evaluó tres dimensiones (claridad, completitud,
accionabilidad) en escala 1-5; SAT es su media aritmética.

Resultados:
- RPA-Baseline:  SAT = 3.09 ± 0.67
- Single-Small:  SAT = 3.93 ± 0.69
- Multi-Small:   SAT = 4.02 ± 0.55
- Multi-Mixed:   SAT = 4.36 ± 0.27

Variante con mayor SAT: Multi-Mixed (SAT=4.36)
Variante con menor SAT: RPA-Baseline (SAT=3.09)

